In [1]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
import os
from dotenv import load_dotenv
from tqdm import tqdm

from eval_utils import vec_ser_evidence_for_sequence_in_file

from langchain_ollama import ChatOllama
from langchain.tools import tool

from ollama import chat
from ollama import ChatResponse

# Load environment variables from .env file
load_dotenv()

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Initialize the ChatOllama model with the specified model name
model_name = 'qwen2.5-coder:7b'

# and initialize the ChatOllama instance
chat_model = ChatOllama(
    model=model_name,
    validate_model_on_init=True,
    temperature=0.7
)

In [3]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

def absoluteFilePaths(directory):
    file_names = []
    file_paths = []
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if f.endswith( ('.mid', '.midi', '.mxl', '.xml', '.musicxml') ):
                file_names.append(f)
                file_paths.append(os.path.abspath(os.path.join(dirpath, f)))
    return file_names, file_paths
# end absoluteFilePaths

hook_file_names, hook_file_paths = absoluteFilePaths(os.getenv('VAL_HOOK'))
gjt_file_names, gjt_file_paths = absoluteFilePaths(os.getenv('VAL_GJT'))

device_name = 'cuda:0'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [4]:
in_seq = 'b_G:7_@2;C:maj_@2'

In [5]:
file_path = gjt_file_paths[2]

In [6]:
evidence = vec_ser_evidence_for_sequence_in_file(
    in_seq,
    file_path,
    tokenizer,
    graph_model=graph_adapter_model,
    bilstm_model=None,
    token_model=token_adapter_model,
    adapter_model=adapter_model,
    max_seq_len=16
)

In [7]:
print(evidence[1]['graph'])
print(evidence[2]['graph'])
print(evidence[3]['graph'])

[0.2883147  0.2883147  0.28041473 0.29641199 0.28008705 0.2883147
 0.28861722 0.1899696  0.18096606 0.18096606 0.31190535 0.31190535
 0.20071159 0.20071159 0.28041473 0.29641202]
[0.53888357 0.57384527 0.58949006 0.54627013 0.6357131  0.5644418
 0.60912466 0.56060445 0.54407322 0.53273803 0.45603859 0.63952571
 0.55093789 0.5622462  0.58949006]
[0.32746568 0.44027913 0.38083833 0.43859181 0.38602456 0.35811272
 0.4355374  0.3581782  0.38435507 0.34429532 0.40381935 0.45614064
 0.3789773  0.44261783]


In [8]:
print(evidence[1]['token'])
print(evidence[2]['token'])
print(evidence[3]['token'])

[-0.21972156 -0.21972156 -0.07495379  0.00134049 -0.18851787 -0.21972156
 -0.0586569   0.13675559  0.09997616  0.09997616  0.03712054  0.03712054
 -0.05885176 -0.05885176 -0.07495379  0.00134049]
[-0.25355247 -0.22695464 -0.0113391  -0.12193452 -0.22903711 -0.16447549
  0.12247895  0.08163381  0.10862888 -0.04475604  0.04887946  0.08769412
 -0.07852297 -0.11398435 -0.0113391 ]
[-0.3393029  -0.2515921  -0.10506023 -0.16589499 -0.18153697 -0.07542764
  0.11478825  0.03438227 -0.13880789 -0.10764454  0.08082932  0.09639615
 -0.18448365 -0.25561368]


In [9]:
print(evidence[1]['adapter'])
print(evidence[2]['adapter'])
print(evidence[3]['adapter'])

[0.41479987 0.41479987 0.29009423 0.3615967  0.29073974 0.41479987
 0.37761283 0.36427748 0.40969878 0.40969878 0.41219455 0.41219455
 0.30036157 0.30036157 0.29009423 0.3615967 ]
[0.38446382 0.46582294 0.49413067 0.50294733 0.47061306 0.46417564
 0.55513144 0.56727576 0.54621929 0.46803111 0.41363096 0.5330441
 0.4463622  0.45714951 0.49413067]
[0.29917079 0.39474988 0.36334467 0.39768523 0.37995195 0.35558549
 0.46340698 0.37086308 0.22712627 0.25900963 0.36706507 0.40149412
 0.27780426 0.34421715]
